<a href="https://colab.research.google.com/github/AbnerJSV/THE_CORE_Trabajos_iA/blob/main/cifar_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#1. Cargar CIFAR‑10
from keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype("float32") / 255
x_test  = x_test.astype("float32") / 255

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

In [8]:
#2.Reescalar imágenes a 96×96
import tensorflow as tf

x_train = tf.image.resize(x_train, (96, 96))
x_test  = tf.image.resize(x_test, (96, 96))

In [ ]:
#3. Separar validación
from sklearn.model_selection import train_test_split

train_X, val_X, train_y, val_y = train_test_split(
    x_train.numpy(), y_train.numpy(), test_size=0.1, random_state=42
)

NameError: name 'x_train' is not defined

In [ ]:
#4. Augmentation mejorado
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

train_generator = train_datagen.flow(train_X, train_y, batch_size=64)

val_datagen = ImageDataGenerator()
val_generator = val_datagen.flow(val_X, val_y, batch_size=64)

NameError: name 'train_X' is not defined

In [ ]:
#5. Transfer Learning con EfficientNetBo
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
output = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#6. Entrenamiento inicial
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    verbose=1
)

NameError: name 'train_generator' is not defined

In [ ]:
import tensorflow as tf

#7. Fine‑tuning real (descongelar TODO EfficientNet)
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    verbose=1
)

In [ ]:
#8. Evaluar
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

In [ ]:
#9. Graficar entrenamiento
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.plot(history_ft.history['accuracy'], label='FT Train Acc')
plt.plot(history_ft.history['val_accuracy'], label='FT Val Acc')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history_ft.history['loss'], label='FT Train Loss')
plt.plot(history_ft.history['val_loss'], label='FT Val Loss')
plt.title("Loss")
plt.legend()

plt.show()